In [1]:
import requests
import pandas as pd
from collections import defaultdict

In [5]:
dataset_url = "https://api.census.gov/data/2024/acs/acs5" # 2024
api_key = "32685a5c4ce6a6c58598b54cce7cf23a6608f8f0"

metadata_url = f"{dataset_url}/variables.json"
meta_response = requests.get(metadata_url)
variables_meta = meta_response.json()["variables"]
var_label_map = {
    var_id: meta["label"].replace("!!", " ")
    for var_id, meta in variables_meta.items()
    if "label" in meta
}

race_api = {
    'B01001A': 'White Alone',
    'B01001B': 'Black or African American Alone',
    'B01001C': 'American Indian and Alaska Native Alone',
    'B01001D': 'Asian Alone',
    'B01001E': 'Native Hawaiian and Other Pacific Islander Alone',
    'B01001F': 'Some Other Race Alone',
    'B01001G': 'Two or More Races',
    'B01001H': 'White Alone, Not Hispanic or Latino',
    'B01001I': 'Hispanic / Latino (Any race)',
}

df = None

for group_id in race_api:
    params = {
        "get": f"group({group_id})",
        "for": "state:26",
        "key": api_key,
    }

    response = requests.get(dataset_url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    group_df = pd.DataFrame(data[1:], columns=data[0])

    # Keep geographic identifiers and this race table's variables.
    columns = ["NAME", "state"] + [
        col for col in group_df.columns
        if col.startswith(f"{group_id}_")
    ]
    group_df = group_df[columns]

    if df is None:
        df = group_df
    else:
        df = df.merge(
            group_df,
            on=["NAME", "state"],
            validate="one_to_one",
        )

df

,NAME,state,B01001A_001E,B01001A_001EA,B01001A_001M,B01001A_001MA,B01001A_002E,B01001A_002EA,B01001A_002M,B01001A_002MA,...,B01001I_029M,B01001I_029MA,B01001I_030E,B01001I_030EA,B01001I_030M,B01001I_030MA,B01001I_031E,B01001I_031EA,B01001I_031M,B01001I_031MA
0,Michigan,26,7431505,None,5859,None,3698749,None,3354,None,...,242,None,5785,None,357,None,2114,None,305,None


In [8]:
group_meta = {}

for group_id in race_api:
    group_response = requests.get(
        f"{dataset_url}/groups/{group_id}.json",
        timeout=30,
    )
    group_response.raise_for_status()
    group_meta.update(group_response.json()["variables"])

cleaned_data_dict = defaultdict(list)
cleaned_data_columns = list()

for column, value in df.iloc[0].items():
    label = group_meta.get(column, {}).get("label", column)
    label = label.replace("!!", " ").replace(":", "")

    table_id = column.split("_")[0]
    race = race_api.get(table_id)

    if race:
        if "Annotation" not in label:
            print(f"{race} - {label}: {value}")
            cleaned_data_dict[race].append(value)
            if label not in cleaned_data_columns:
                cleaned_data_columns.append(label)
    else:
        print(f"{label}: {value}")

Geographic Area Name: Michigan
state: 26
White Alone - Estimate Total: 7431505
White Alone - Margin of Error Total: 5859
White Alone - Estimate Total Male: 3698749
White Alone - Margin of Error Total Male: 3354
White Alone - Estimate Total Male Under 5 years: 180325
White Alone - Margin of Error Total Male Under 5 years: 725
White Alone - Estimate Total Male 5 to 9 years: 196006
White Alone - Margin of Error Total Male 5 to 9 years: 2433
White Alone - Estimate Total Male 10 to 14 years: 213147
White Alone - Margin of Error Total Male 10 to 14 years: 2578
White Alone - Estimate Total Male 15 to 17 years: 135212
White Alone - Margin of Error Total Male 15 to 17 years: 639
White Alone - Estimate Total Male 18 and 19 years: 91026
White Alone - Margin of Error Total Male 18 and 19 years: 534
White Alone - Estimate Total Male 20 to 24 years: 236137
White Alone - Margin of Error Total Male 20 to 24 years: 732
White Alone - Estimate Total Male 25 to 29 years: 231990
White Alone - Margin of Err

In [7]:
cleaned_data_dict = dict(cleaned_data_dict)

cleaned_data = pd.DataFrame(cleaned_data_dict.values(), columns=cleaned_data_columns)
cleaned_data.insert(0, "Race", cleaned_data_dict.keys(), True)
cleaned_data

,Race,Estimate Total,Margin of Error Total,Estimate Total Male,Margin of Error Total Male,Estimate Total Male Under 5 years,Margin of Error Total Male Under 5 years,Estimate Total Male 5 to 9 years,Margin of Error Total Male 5 to 9 years,Estimate Total Male 10 to 14 years,...,Estimate Total Female 45 to 54 years,Margin of Error Total Female 45 to 54 years,Estimate Total Female 55 to 64 years,Margin of Error Total Female 55 to 64 years,Estimate Total Female 65 to 74 years,Margin of Error Total Female 65 to 74 years,Estimate Total Female 75 to 84 years,Margin of Error Total Female 75 to 84 years,Estimate Total Female 85 years and over,Margin of Error Total Female 85 years and over
0,White Alone,7431505,5859,3698749,3354,180325,725,196006,2433,213147,...,446762,970,551554,900,486125,786,247998,1954,112982,1950
1,Black or African American Alone,1337239,6247,634240,3257,41985,991,46556,1573,48729,...,88557,730,84136,791,68404,636,30881,839,13681,772
2,American Indian and Alaska Native Alone,45929,1849,22850,1032,1131,234,1533,208,1865,...,2747,270,3060,292,2296,186,962,121,234,110
3,Asian Alone,345417,2734,170929,1657,8763,429,9698,703,10389,...,25388,455,17032,363,11262,431,5790,381,2020,357
4,Native Hawaiian and Other Pacific Islander Alone,3032,409,1987,298,175,105,146,104,1,...,91,56,90,71,8,7,21,17,4,6
5,Some Other Race Alone,194572,5227,101142,3111,7170,791,7860,693,8652,...,10929,825,7703,669,7367,849,2439,426,1285,535
6,Two or More Races,720067,9241,363822,5218,37436,1250,38526,1599,36435,...,34875,1233,30165,1138,18140,860,7785,484,2589,426
7,"White Alone, Not Hispanic or Latino",7291631,4818,3628942,2699,173727,484,189017,2462,205434,...,439910,652,547351,774,483798,649,246929,1932,112421,1929
8,Hispanic / Latino (Any race),592045,30,301692,302,24709,222,28016,824,28748,...,31792,353,21756,188,12973,242,5785,357,2114,305
